In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding,LSTM,Dropout,Bidirectional,Dropout

In [ ]:
df=pd.read_csv('/content/fake_news_dataset.csv')

In [ ]:
df

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake
...,...,...,...,...,...,...,...
19995,House party born.,hit and television I change very our happy doo...,2024-12-04,BBC,Gary Miles,Entertainment,fake
19996,Though nation people maybe price box.,fear most meet rock even sea value design stan...,2024-05-26,Daily News,Maria Mcbride,Entertainment,real
19997,Yet exist with experience unit.,activity loss very provide eye west create wha...,2023-04-17,BBC,Kristen Franklin,Entertainment,real
19998,School wide itself item.,term point general common training watch respo...,2024-06-30,Reuters,David Wise,Health,fake


In [ ]:
df.head()

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake


In [ ]:
df['label'].value_counts()

,count
label,
fake,10056
real,9944


In [ ]:
#replace missing titles with empty string
df['title']=df['title'].fillna(' ')
#replace missing article text with empty string
df['text']=df['text'].fillna(' ')
#combine title and article text into one column
df['content']=df['title']+' '+df['text']
#convert fake label ito 0 and real label into 1
df['label']=df['label'].str.lower().map({'fake':0,'real':1})

In [ ]:
#remove rows where the label could not be converted
df=df.dropna(subset=['label'])
x=df['content']
y=df['label'].astype(int)

In [ ]:
x.head()

,content
0,Foreign Democrat final. more tax development b...
1,To offer down resource great point. probably g...
2,Himself church myself carry. them identify for...
3,You unit its should. phone which item yard Rep...
4,Billion believe employee summer how. wonder my...


In [ ]:
y.head()

,label
0,1
1,0
2,0
3,0
4,0


In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
#create a tokenizer that keeps a maximum of 20000 words
#oov_token handles words that are not in the vocabulary
tokenizer=Tokenizer(num_words=20000,oov_token='<OOV>')
#fit the tokenizer on the training data and it learn the word vocabulary using only training data
tokenizer.fit_on_texts(x_train)
#convert training text into numerical sequences using distinct variable names to avoid overwriting
train_sequences=tokenizer.texts_to_sequences(x_train)
#convert test text into numerical sequences
test_sequences=tokenizer.texts_to_sequences(x_test)

In [ ]:
#set maximum no of tokens in each article
max_len=300
#make every training article exactly 300 tokens long
#padding='post' adds zeros at the end
#truncating='post' removes extra tokens from the end
x_train=pad_sequences(train_sequences,maxlen=max_len,padding='post',truncating='post')
#make every test article exactly 300 tokens long
x_test=pad_sequences(test_sequences,maxlen=max_len,padding='post',truncating='post')

In [ ]:
#create empty sequential model
model=Sequential()

#convert each word id into a vector of 64 numbers
#20000 max vocabulary size
#64=embedding dimension
model.add(Embedding(20000,64))

#bilstm reads article in 2 directions
#forward reads left to right
#backward reads right to left
#64=no of lstm units in each direction
model.add(Bidirectional(LSTM(64)))

#randomly switch of 30% of units during training
#it helps to reduce overfitting
#model.add(Dropout(0.3))

#dense layer with 256 units
model.add(Dense(32,activation='relu'))
model.add(Dense(1,activation='sigmoid'))
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
history=model.fit(x_train,y_train,epochs=10,batch_size=64,validation_split=0.2)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.4938 - loss: 0.6935 - val_accuracy: 0.5056 - val_loss: 0.6931
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.5295 - loss: 0.6911 - val_accuracy: 0.4884 - val_loss: 0.6956
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.5838 - loss: 0.6730 - val_accuracy: 0.4919 - val_loss: 0.7078
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.6177 - loss: 0.6488 - val_accuracy: 0.5013 - val_loss: 0.7212
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.6490 - loss: 0.6233 - val_accuracy: 0.4884 - val_loss: 0.7427
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.6749 - loss: 0.5947 - val_accuracy: 0.4916 - val_loss: 0.7996
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.6973 - loss: 0.5600 - val_accuracy: 0.4913 - val_loss: 0.8345
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.7291 - loss: 0.5190 - val_accu

In [ ]:
loss,accuracy=model.evaluate(x_test,y_test)
print('Test accuracy:',accuracy)
print('Test loss:',loss)

125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5033 - loss: 1.2983
Test accuracy: 0.503250002861023
Test loss: 1.298323392868042


In [ ]:
y_probability=model.predict(x_test)
y_pred=(y_probability>=0.5).astype(int).ravel()
print("Predictions:",y_pred[:10])
print("Actual     :",y_test.values[:10])

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
Predictions: [1 1 0 0 0 0 1 1 1 1]
Actual     : [1 0 0 1 0 0 1 0 1 1]


In [ ]:
sample_news="""A technology startup unveiled a smartphone battery that can power a device continuously for 100 years without recharging.
Company representatives explained that the battery uses a secret energy source not yet known to science. Industry analysts believe
the invention will immediately replace all existing battery technologies and make electricity bills obsolete."""
#convert new article into numerical tokens
sample_seq=tokenizer.texts_to_sequences([sample_news])
#make the sentence 300 tokens long
sample_pad=pad_sequences(sample_seq,maxlen=max_len,padding='post',truncating='post')
#make prediction
prediction=model.predict(sample_pad)
#display predicted probability
print("Predicted probability:",prediction[0][0])
#if probabilityis 0.5 or higher,predict real news
if prediction[0][0]>=0.5:
  print("Predicted label: Real")
else:
  print("Predicted label: Fake")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Predicted probability: 0.4974886
Predicted label: Fake
